In [1]:
tabla ='qtiaa10'
col_tabla = 'solopefec'
essi='essi_dat_cqx006_etl'
col_essi='sol_fec'
dat= "dat_ceqx006_essi"
col_dat='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [6]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [7]:

fecha= '2023-05-01'

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

cqxaneste= pd.read_sql_query(f"SELECT id_anestesia,cod_ane FROM dim_cqxaneste", con=connection2)
cqxaneste['cod_ane']=cqxaneste['cod_ane'].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)


In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(32959, 14)

In [11]:
#Eliminamos las columnas que no se usarán aquí: las descripciones previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'des_ane']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [12]:
base1.shape

(32959, 11)

In [13]:
inicioProceso=time.time()

In [14]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [15]:
base1.shape

(32959, 11)

In [16]:
control_a.append(base1.shape[0])

In [17]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(32959, 12)

In [18]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [19]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(32959, 12)

In [20]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [21]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(32959, 12)

In [22]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [23]:
merge.shape

(32959, 12)

In [24]:
cqxaneste=cqxaneste.rename(columns={"id_anestesia": "id_anest"})
base1['cod_ane']=base1['cod_ane'].str.strip()
base1["cod_ane"]=base1["cod_ane"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxaneste,how='inner',on='cod_ane')
base1=pd.merge(base1,cqxaneste,how='left',on='cod_ane')
base1.shape

(32959, 12)

In [25]:
merge.shape

(32959, 12)

In [26]:
log_falla('id_anest', 'cod_ane', True)
log_control('dim_cqxaneste')
base1=base1.drop('cod_ane',axis=1)

In [27]:
pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(32959, 12)

In [28]:
merge.shape

(32345, 12)

In [29]:
log_falla('id_pacsec', 'pac_sec', True)
base1=base1.drop('pac_sec',axis=1)
dim.append('dim_pacsec')
control_d.append(base1.shape[0])

In [30]:
base1['sol_fec_temp'] = base1['sol_fec'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"sol_fec_temp"})
tiempo["sol_fec_temp"] = tiempo['sol_fec_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='sol_fec_temp')
base1 = pd.merge(base1, tiempo, how='left', on='sol_fec_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_sol","dt_ano":"id_ano_sol","dt_mes":"id_mes_sol",
                            "dt_dia":"id_dia_sol","dt_dia_sem":"id_diasem_sol","dt_sem":"id_sem_sol"})
base1=base1.drop("sol_fec_temp",axis=1)
base1.shape

(32959, 17)

In [31]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32959 entries, 0 to 32958
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sol_num        32959 non-null  float64
 1   inf_ane        32959 non-null  float64
 2   sec_ane        32959 non-null  float64
 3   est_reg        32959 non-null  object 
 4   sol_fec        32959 non-null  object 
 5   act_med        32959 non-null  float64
 6   id_oricas      32959 non-null  int64  
 7   id_cas         32959 non-null  int64  
 8   id_red         32959 non-null  int64  
 9   id_anest       32959 non-null  int64  
 10  id_pacsec      32345 non-null  float64
 11  id_time_sol    32958 non-null  float64
 12  id_mes_sol     32958 non-null  float64
 13  id_dia_sol     32958 non-null  float64
 14  id_diasem_sol  32958 non-null  float64
 15  id_sem_sol     32958 non-null  float64
 16  id_ano_sol     32958 non-null  float64
dtypes: float64(11), int64(4), object(2)
memory usage: 

In [32]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [33]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [34]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [35]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [36]:
base

,sol_num,id_sem_sol,id_oricas,sol_fec,id_cas,est_reg,id_pacsec,sec_ane,id_dia_sol,act_med,id_ano_sol,id_time_sol,id_diasem_sol,id_anest,id_red,id_mes_sol,inf_ane
0,3828.0,1.0,1,2023-05-09,370,1,43637583.0,1.0,9.0,6708687.0,2023.0,20230509.0,2.0,1,1,5.0,1.0
1,60105.0,2.0,1,2023-10-17,370,1,56548964.0,1.0,17.0,9341987.0,2023.0,20231017.0,2.0,3,1,10.0,1.0
2,61246.0,1.0,1,2024-03-13,370,1,53944936.0,1.0,13.0,9464723.0,2024.0,20240313.0,3.0,4,1,3.0,1.0
3,78053.0,1.0,1,2023-05-01,370,1,54697310.0,1.0,1.0,10090155.0,2023.0,20230501.0,1.0,4,1,5.0,1.0
4,74627.0,1.0,1,2023-05-02,370,1,41462630.0,1.0,2.0,9870249.0,2023.0,20230502.0,2.0,4,1,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32954,15436.0,1.0,1,2023-06-26,275,1,41434943.0,1.0,26.0,2226115.0,2023.0,20230626.0,1.0,17,21,6.0,1.0
32955,15461.0,1.0,1,2023-06-22,275,1,54055836.0,1.0,22.0,2230333.0,2023.0,20230622.0,4.0,17,21,6.0,1.0
32956,15429.0,1.0,1,2023-06-21,275,1,46932944.0,1.0,21.0,2223875.0,2023.0,20230621.0,3.0,17,21,6.0,1.0
32957,15420.0,1.0,1,2023-06-12,275,1,48693840.0,1.0,12.0,2224148.0,2023.0,20230612.0,1.0,17,21,6.0,1.0


In [37]:
base.columns

Index(['sol_num', 'id_sem_sol', 'id_oricas', 'sol_fec', 'id_cas', 'est_reg',
       'id_pacsec', 'sec_ane', 'id_dia_sol', 'act_med', 'id_ano_sol',
       'id_time_sol', 'id_diasem_sol', 'id_anest', 'id_red', 'id_mes_sol',
       'inf_ane'],
      dtype='object')

In [38]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

3959

In [39]:
fecha_fin = "2024-12-31"

In [40]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [41]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx006_essi,2023-06-27,16:57,16:59,dim_oricas,2023-05-01,2024-12-31,32959,32959,0,0,id_oricas
1,13,DAT,dat_ceqx006_essi,2023-06-27,16:57,16:59,dim_cas,2023-05-01,2024-12-31,32959,32959,0,0,id_cas
2,13,DAT,dat_ceqx006_essi,2023-06-27,16:57,16:59,dim_red,2023-05-01,2024-12-31,32959,32959,0,0,id_red
3,13,DAT,dat_ceqx006_essi,2023-06-27,16:57,16:59,dim_cqxaneste,2023-05-01,2024-12-31,32959,32959,0,0,id_anest
4,13,DAT,dat_ceqx006_essi,2023-06-27,16:57,16:59,dim_pacsec,2023-05-01,2024-12-31,32959,32959,0,0,id_pacsec


In [42]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permiso denegado a la tabla logs

[SQL: INSERT INTO logs (codigo, proceso, dat, fecha_ejecucion, hora_inicio, hora_fin, dim, fecha_ini, fecha_ter, esperado, obtenido, faltante, falla, id_afectado) VALUES (%(codigo)s, %(proceso)s, %(dat)s, %(fecha_ejecucion)s, %(hora_inicio)s, %(hora_fin)s, %(dim)s, %(fecha_ini)s, %(fecha_ter)s, %(esperado)s, %(obtenido)s, %(faltante)s, %(falla)s, %(id_afectado)s)]
[parameters: ({'codigo': 13, 'proceso': 'DAT', 'dat': 'dat_ceqx006_essi', 'fecha_ejecucion': '2023-06-27', 'hora_inicio': '16:57', 'hora_fin': '16:59', 'dim': 'dim_oricas', 'fecha_ini': '2023-05-01', 'fecha_ter': '2024-12-31', 'esperado': 32959, 'obtenido': 32959, 'faltante': 0, 'falla': 0, 'id_afectado': 'id_oricas'}, {'codigo': 13, 'proceso': 'DAT', 'dat': 'dat_ceqx006_essi', 'fecha_ejecucion': '2023-06-27', 'hora_inicio': '16:57', 'hora_fin': '16:59', 'dim': 'dim_cas', 'fecha_ini': '2023-05-01', 'fecha_ter': '2024-12-31', 'esperado'

In [ ]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 737.712 segundos


In [ ]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [ ]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()